# Calibration Audit (`actual` vs. `held` Supplied Year) & Layer-Geometry Diagnostics

Zero-inference, CPU-only diagnostic bundle covering:
1. **Outcome-grounded calibration audit (`actual` vs. `held` supplied-year contrast)** against observed mortality ($Y$) for Work 1 (same-year SUS, 7 trailing-window models `2007-2013` through `2014`), Work 7 (prospective SUS Scope A, 6 trailing-window models `2007-2012`), and Work 6b (FLChain `H = 1095`, threshold-free metrics), with multiplicity counts.
2. **Work 8 layer-geometry micro-checks:**
   - Partial `PC1 x year` controlling for linear and natural-cubic-spline predicted logit (`L0..L11`) plus FLChain `2002` vs. `2003` split (`n=46` subsampling).
   - Threshold-free continuous TCAV saturation (`mean_tcav_binary_entropy_nats`, `mean_abs_dev_from_half`, `exact_endpoint_share`, MC SE, and equal-`n` subsample controls) matching Work 8's exact per-`(stratum, layer)` RNG stream.
   - Analytical random-gradient null resultant floor ($c_{192}/\sqrt{n}$) + single `R` identity check + within-dataset non-nested dispersion summary.
   - Input-feature correlation-matrix participation ratio (with column-permutation null, `log1p` SUS sensitivity, and 6-feature FLChain excluding `creatinine` plus `creatinine` missingness table/sensitivity) paired against same-row standardized embedding PR at `L0`, `L9`, and `L11`.

All printed summaries and figures are strictly aggregated across patient strata; no row-level SUS tables or patient scatters appear in outputs.

In [1]:
import json
import resource
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import calibration_and_layer_diagnostics as diag
import concept_subgroup_drift_pilot as pilot
import flchain_probe as probe
import layer_compression as lc
import prospective_audit as pa
import tcav_geometry as geometry

In [2]:
OUTPUT_DIR = Path("stats/calibration_and_layer_diagnostics")
FIG_DIR = OUTPUT_DIR / "figures"
BOOTSTRAP_DRAWS = diag.BOOTSTRAP_DRAWS
PERM_NULL_DRAWS = diag.PERM_NULL_DRAWS
SUBSAMPLE_REPEATS = diag.SUBSAMPLE_REPEATS

WORK1_PRIVATE_DIR = Path("stats/concept_agreement_study/extensions/temporal_input_attribution/generated_state/work1_v5_primary/private")
WORK1_AGG_PATH = Path("stats/concept_agreement_study/extensions/temporal_input_attribution/generated_state/work1_v5_primary/aggregate.json")
WORK1_PROV_PATH = Path("stats/concept_agreement_study/extensions/temporal_input_attribution/checks_cpu_primary/provenance.json")
POPULATION_PATH = Path("stats/temporal_robustness/_population/prepared.pkl")
WORK7_PERF_PATH = Path("stats/prospective_target_audit/scope_a/longitudinal_performance.csv")
WORK7_SENS_PATH = Path("stats/prospective_target_audit/scope_a/year_input_sensitivity.csv")
WORK7_META_PATH = Path("stats/prospective_target_audit/scope_a/prospective_splits_and_metadata.npz")
FLCHAIN_CSV_PATH = Path("data/flchain/free_light_chain_mortality.csv")
FLCHAIN_SPLITS_PATH = Path("stats/flchain_last_layer_probe/splits_and_predictions.npz")
FLCHAIN_SENS_PATH = Path("stats/flchain_last_layer_probe/sample_yr_sensitivity.csv")
SUS_LAYER_DIR = Path("stats/sus_layer_compression")
SUS_BY_YEAR_DIR = Path("analysis_outputs/sus_layer_compression/by_year")
FLCHAIN_LAYER_DIR = Path("stats/flchain_layer_compression")
FLCHAIN_BY_YEAR_DIR = Path("analysis_outputs/flchain_layer_compression/by_year")

## 1. Run Metadata & Input SHA-256 Verification

In [3]:
run_start_utc = datetime.now(timezone.utc).isoformat()
run_start_mono = time.monotonic()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

CHECKS = []
INPUT_SHA256 = {}

def record_check(check_name: str, value, rule: str, passed: bool):
    CHECKS.append({
        "check": str(check_name),
        "value": str(value),
        "rule": str(rule),
        "passed": bool(passed),
    })

def register_sha256(label: str, path: Path) -> str:
    sha = probe.file_sha256(path)
    INPUT_SHA256[label] = {"path": str(path), "sha256": sha}
    return sha

pop_sha = register_sha256("sus_population_prepared_pkl", POPULATION_PATH)
fl_sha = register_sha256("flchain_raw_csv", FLCHAIN_CSV_PATH)
record_check("sus_prepared_pkl_sha256", pop_sha == diag.POPULATION_SHA256, "matches pinned SHA-256", pop_sha == diag.POPULATION_SHA256)
record_check("flchain_csv_sha256", fl_sha == diag.FLCHAIN_CSV_SHA256, "matches pinned SHA-256", fl_sha == diag.FLCHAIN_CSV_SHA256)

for label, p in [
    ("work1_provenance_json", WORK1_PROV_PATH),
    ("work1_aggregate_json", WORK1_AGG_PATH),
    ("work7_longitudinal_perf_csv", WORK7_PERF_PATH),
    ("work7_sensitivity_csv", WORK7_SENS_PATH),
    ("work7_metadata_npz", WORK7_META_PATH),
    ("flchain_splits_npz", FLCHAIN_SPLITS_PATH),
    ("flchain_sensitivity_csv", FLCHAIN_SENS_PATH),
    ("sus_layer_arrays_npz", SUS_LAYER_DIR / "layer_arrays.npz"),
    ("sus_layer_curves_csv", SUS_LAYER_DIR / "layer_curves.csv"),
    ("sus_layer_geometry_csv", SUS_LAYER_DIR / "layer_geometry.csv"),
    ("sus_leading_dir_corr_csv", SUS_BY_YEAR_DIR / "leading_direction_year_correlation.csv"),
    ("flchain_layer_arrays_npz", FLCHAIN_LAYER_DIR / "layer_arrays.npz"),
    ("flchain_layer_curves_csv", FLCHAIN_LAYER_DIR / "layer_curves.csv"),
    ("flchain_layer_geometry_csv", FLCHAIN_LAYER_DIR / "layer_geometry.csv"),
    ("flchain_curves_by_year_csv", FLCHAIN_BY_YEAR_DIR / "layer_curves_by_year.csv"),
]:
    register_sha256(label, p)

work1_prov = json.loads(WORK1_PROV_PATH.read_text(encoding="utf-8"))
work1_agg = json.loads(WORK1_AGG_PATH.read_text(encoding="utf-8"))
for sys_entry in work1_prov["systems"]:
    stem = f"{sys_entry['build']}_{int(sys_entry['reference'])}"
    npz_path = WORK1_PRIVATE_DIR / f"{stem}.npz"
    register_sha256(f"work1_private_{stem}", npz_path)

git_head = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
git_status_lines = subprocess.check_output(["git", "status", "--porcelain"], text=True).splitlines()
print(f"Verified {len(INPUT_SHA256)} inputs on CPU | git_head={git_head[:10]} | draws={BOOTSTRAP_DRAWS}")

Verified 24 inputs on CPU | git_head=16425eebff | draws=500


## 2. Part 1A — Work 1 Same-Year SUS Calibration Audit (`last_3`, `2007-2014`)

In [4]:
population = pa.load_population(POPULATION_PATH, diag.POPULATION_SHA256)

systems_sorted = sorted(work1_prov["systems"], key=lambda s: int(s["reference"]))
all_last3 = all("last_3" in s["variants"] for s in systems_sorted)
record_check("work1_all_7_systems_have_last_3_variant", all_last3 and len(systems_sorted) == 7, "== True for all 7", all_last3 and len(systems_sorted) == 7)

work1_base_rates = {}
work1_thresholds = {}
for s in systems_sorted:
    ref_y = int(s["reference"])
    roles = pilot.reference_year_roles(population, ref_y, 42)
    sel_mask = (population["year"] == ref_y) & np.isin(
        roles["row_role"], ("rule_discovery", "cav_fit", "concept_selection")
    )
    work1_base_rates[ref_y] = float(population["y"][sel_mask].mean())
    work1_thresholds[ref_y] = float(s["death_threshold"]["last_3"])

print("Work 1 frozen reference base rates & shared thresholds ready for 7 trailing-window systems.")

Work 1 frozen reference base rates & shared thresholds ready for 7 trailing-window systems.


### Work 1 Calibration Metrics, Paired Bootstrap & `aggregate.json` Point-Estimate Parity

In [5]:
agg_lookup = {}
for case in work1_agg["cases"]:
    key = (case["build"], int(case["reference"]), int(case["year"]))
    if key not in agg_lookup:
        agg_lookup[key] = case["cells"]

work1_frames = []
work1_max_parity_diff = 0.0
work1_parity_count = 0
work1_clipped_total = 0

metric_parity_map = [
    ("mean_probability", "mean_probability"),
    ("mean_entropy_nats", "mean_entropy"),
    ("brier", "brier"),
    ("auroc", "auroc"),
    ("recall", "death_recall"),
    ("false_alarm_rate", "false_positive_rate"),
    ("death_f1", "death_f1"),
]

for s in systems_sorted:
    build_name = s["build"]
    ref_y = int(s["reference"])
    stem = f"{build_name}_{ref_y}"
    arr = np.load(WORK1_PRIVATE_DIR / f"{stem}.npz", allow_pickle=False)
    yrs = arr["year"].astype(np.int64)
    keep = yrs <= diag.SUS_LAST_OUTCOME_YEAR
    y_all = arr["label"][keep].astype(np.float64)
    pids_all = arr["patient_id"][keep].astype(str)
    yrs_kept = yrs[keep]
    base_rate = work1_base_rates[ref_y]
    thr = work1_thresholds[ref_y]

    slices = [("later_years", yrs_kept > ref_y, False)]
    for yr_val in sorted(np.unique(yrs_kept).tolist()):
        slices.append((str(yr_val), yrs_kept == yr_val, int(yr_val) == ref_y))

    for scaling in ("consistent", "historical"):
        pa_full = arr[f"{scaling}_actual_probability"][keep].astype(np.float64)
        ph_full = arr[f"{scaling}_held_probability"][keep].astype(np.float64)
        for stratum_name, mask, is_ref in slices:
            boot_df = diag.paired_calibration_bootstrap(
                y_all[mask], pa_full[mask], ph_full[mask], pids_all[mask],
                base_rate=base_rate, threshold=thr, draws=BOOTSTRAP_DRAWS, seed=diag.BOOTSTRAP_SEED,
            )
            work1_clipped_total += int(boot_df["clipped_rows"].max())
            boot_df.insert(0, "reference_year", ref_y)
            boot_df.insert(1, "model_window", "last_3")
            boot_df.insert(2, "scaling", scaling)
            boot_df.insert(3, "stratum", stratum_name)
            boot_df.insert(4, "is_reference_year_contrast", bool(is_ref))
            boot_df.insert(5, "base_rate", base_rate)
            boot_df.insert(6, "shared_threshold", thr)
            boot_df.insert(7, "threshold_origin", "historical_ref_death_f1_shared_across_scalings")
            work1_frames.append(boot_df)

            if stratum_name != "later_years":
                yr_int = int(stratum_name)
                agg_cells = agg_lookup.get((build_name, ref_y, yr_int))
                if agg_cells is not None:
                    for quantity, cell_suffix in (("actual", "actual"), ("held", "held")):
                        ref_cell = agg_cells[f"{scaling}/{cell_suffix}"]
                        sub_q = boot_df[boot_df["quantity"] == quantity].set_index("metric")
                        for new_m, old_m in metric_parity_map:
                            v_new = float(sub_q.loc[new_m, "estimate"])
                            v_old = float(ref_cell[old_m]["estimate"])
                            if np.isfinite(v_new) and np.isfinite(v_old):
                                work1_max_parity_diff = max(work1_max_parity_diff, abs(v_new - v_old))
                                work1_parity_count += 1

work1_cal_df = pd.concat(work1_frames, ignore_index=True)
work1_cal_df.to_csv(OUTPUT_DIR / "calibration_audit_work1.csv", index=False)

record_check("work1_zero_clipped_probabilities_through_2014", work1_clipped_total, "== 0", work1_clipped_total == 0)
record_check("work1_point_parity_vs_aggregate_json", work1_max_parity_diff, f"<= {diag.PARITY_ATOL} (n={work1_parity_count})", work1_max_parity_diff <= diag.PARITY_ATOL)
print(f"Work 1 calibration: rows={len(work1_cal_df)} | parity_checks={work1_parity_count} | max_diff={work1_max_parity_diff:.2e} | clipped={work1_clipped_total}")

Work 1 calibration: rows=5040 | parity_checks=980 | max_diff=2.22e-16 | clipped=0


## 3. Part 1B — Work 7 Prospective SUS Calibration Reconciliation

In [6]:
w7_perf = pd.read_csv(WORK7_PERF_PATH)
w7_sens = pd.read_csv(WORK7_SENS_PATH)
work7_cal_df = diag.reconcile_prospective_scope_a_calibration(w7_perf, w7_sens)
work7_cal_df.to_csv(OUTPUT_DIR / "calibration_audit_work7.csv", index=False)

# Verify actual values match longitudinal_performance.csv exactly
m_chk = w7_perf.merge(
    work7_cal_df[work7_cal_df["quantity"] == "actual"],
    on=["reference_year", "scaling", "input_year", "stratum", "metric"],
    suffixes=("_src", "_out"),
)
w7_max_diff = float(np.max(np.abs(m_chk["estimate_src"] - m_chk["estimate_out"])))
ll_ok = bool((work7_cal_df.loc[work7_cal_df["metric"] == "log_loss", "status"] == "unavailable_no_per_row_probs").all())
held_ok = bool((work7_cal_df.loc[(work7_cal_df["quantity"] == "held") & (work7_cal_df["metric"] == "brier"), "status"] == "point_only_recovered_from_csv").all())
record_check("work7_actual_parity_vs_csv", w7_max_diff, f"<= {diag.PARITY_ATOL}", w7_max_diff <= diag.PARITY_ATOL)
record_check("work7_unavailable_log_loss_and_point_only_held_marked", ll_ok and held_ok, "True", ll_ok and held_ok)
print(f"Work 7 reconciliation: rows={len(work7_cal_df)} | max_diff={w7_max_diff:.2e}")

Work 7 reconciliation: rows=10920 | max_diff=0.00e+00


## 4. Part 1C — Work 6b FLChain (`H = 1095`) Threshold-Free Calibration Audit & Multiplicity Summary

In [7]:
fl_splits = np.load(FLCHAIN_SPLITS_PATH, allow_pickle=False)
fl_eval_idx = fl_splits["h1095_eval_idx"]
fl_ref_fit_idx = fl_splits["h1095_ref_fit_idx"]
fl_ref_eval_idx = fl_splits["h1095_ref_eval_idx"]
fl_future_idx = fl_splits["h1095_future_idx"]
fl_y_all = fl_splits["h1095_y"]
fl_yr_all = fl_splits["h1095_sample_yr"]
fl_sid_all = fl_splits["h1095_subject_id"]

fl_base_rate = float(fl_y_all[fl_ref_fit_idx].mean())
record_check("flchain_ref_fit_base_rate", fl_base_rate, "== 27/340", abs(fl_base_rate - 27.0 / 340.0) <= 1e-12)

eval_y = fl_y_all[fl_eval_idx].astype(np.float64)
eval_yr = fl_yr_all[fl_eval_idx].astype(np.int64)
eval_sid = fl_sid_all[fl_eval_idx].astype(str)

fl_slices = [
    ("1997_eval", np.isin(fl_eval_idx, fl_ref_eval_idx), True),
    ("1998-2003", np.isin(fl_eval_idx, fl_future_idx), False),
    ("1998-2001", np.isin(fl_eval_idx, fl_future_idx) & np.isin(eval_yr, [1998, 1999, 2000, 2001]), False),
    ("2002-2003", np.isin(fl_eval_idx, fl_future_idx) & np.isin(eval_yr, [2002, 2003]), False),
]
for yr_val in (1998, 1999, 2000, 2001, 2002, 2003):
    fl_slices.append((str(yr_val), np.isin(fl_eval_idx, fl_future_idx) & (eval_yr == yr_val), False))

fl_sens_csv = pd.read_csv(FLCHAIN_SENS_PATH)
fl_sens_h1095 = fl_sens_csv[fl_sens_csv["target"] == "horizon_1095d"].copy()

fl_frames = []
fl_max_parity_diff = 0.0
fl_parity_count = 0
fl_clipped_total = 0

for scaling in ("consistent", "historical"):
    pa_fl = fl_splits[f"h1095_{scaling}_p_actual"].astype(np.float64)
    ph_fl = fl_splits[f"h1095_{scaling}_p_held"].astype(np.float64)
    for stratum_name, mask, is_ref in fl_slices:
        boot_df = diag.paired_calibration_bootstrap(
            eval_y[mask], pa_fl[mask], ph_fl[mask], eval_sid[mask],
            base_rate=fl_base_rate, threshold=None, draws=BOOTSTRAP_DRAWS, seed=diag.BOOTSTRAP_SEED,
        )
        fl_clipped_total += int(boot_df["clipped_rows"].max())
        boot_df.insert(0, "target", "horizon_1095d")
        boot_df.insert(1, "scaling", scaling)
        boot_df.insert(2, "stratum", stratum_name)
        boot_df.insert(3, "is_reference_year_contrast", bool(is_ref))
        boot_df.insert(4, "base_rate", fl_base_rate)
        fl_frames.append(boot_df)

        src = fl_sens_h1095[(fl_sens_h1095["scaling"] == scaling) & (fl_sens_h1095["stratum"] == stratum_name)]
        if len(src) == 1:
            srow = src.iloc[0]
            sub_map = boot_df.set_index(["quantity", "metric"])["estimate"]
            for new_m, old_prefix in [("mean_entropy_nats", "mean_entropy"), ("mean_probability", "mean_probability"), ("brier", "brier"), ("auroc", "auroc")]:
                for q, suf in [("actual", "actual"), ("held", "held"), ("actual_minus_held", "diff")]:
                    v_new = float(sub_map.loc[(q, new_m)])
                    v_old = float(srow[f"{old_prefix}_{suf}"])
                    if np.isfinite(v_new) and np.isfinite(v_old):
                        fl_max_parity_diff = max(fl_max_parity_diff, abs(v_new - v_old))
                        fl_parity_count += 1

flchain_cal_df = pd.concat(fl_frames, ignore_index=True)
flchain_cal_df.to_csv(OUTPUT_DIR / "calibration_audit_flchain.csv", index=False)

record_check("flchain_zero_clipped_probabilities", fl_clipped_total, "== 0", fl_clipped_total == 0)
record_check("flchain_point_parity_vs_sensitivity_csv", fl_max_parity_diff, f"<= {diag.PARITY_ATOL} (n={fl_parity_count})", fl_max_parity_diff <= diag.PARITY_ATOL)

mult_df = diag.multiplicity_summary({
    "work1_same_year": work1_cal_df,
    "work7_prospective": work7_cal_df,
    "work6b_flchain": flchain_cal_df,
})
mult_df.to_csv(OUTPUT_DIR / "multiplicity_summary.csv", index=False)
print(f"FLChain calibration: rows={len(flchain_cal_df)} | parity_checks={fl_parity_count} | max_diff={fl_max_parity_diff:.2e}")
print(mult_df.to_string(index=False))

FLChain calibration: rows=840 | parity_checks=192 | max_diff=1.30e-16
            table  total_intervals  excluding_zero  share_excluding_zero  chance_expected_at_alpha
  work1_same_year             4200            3165              0.753571                     210.0
work7_prospective             5032            3322              0.660175                     251.6
   work6b_flchain              732             513              0.700820                      36.6
              ALL             9964            7000              0.702529                     498.2


## 5. Part 2a — Partial `PC1 x Year` Controlling for Linear & Natural-Spline Predicted Logit (`L0..L11`)

In [8]:
sus_arr = np.load(SUS_LAYER_DIR / "layer_arrays.npz", allow_pickle=False)
fl_arr = np.load(FLCHAIN_LAYER_DIR / "layer_arrays.npz", allow_pickle=False)
fl_arr_sample_yr = fl_yr_all[fl_arr["eval_row_index"]].astype(np.int64)

sus_corr_csv = pd.read_csv(SUS_BY_YEAR_DIR / "leading_direction_year_correlation.csv")

pc1_frames = []
pc1_max_parity_diff = 0.0
pc1_parity_count = 0

sus_build_configs = [
    ("same_year_consistent", "sus_same_year", "same_year_last_3_2011", "consistent", "input_years_2011_2013"),
    ("same_year_historical", "sus_same_year", "same_year_last_3_2011", "historical", "input_years_2011_2013"),
    ("prospective_consistent", "sus_prospective", "prospective_last_3_2011", "consistent", "input_years_2011_2013"),
]

for key, ds_label, csv_build, csv_scaling, csv_stratum in sus_build_configs:
    emb_all = sus_arr[f"{key}__target_embedding"]
    yr_all = sus_arr[f"{key}__year"].astype(np.int64)
    prob_all = sus_arr[f"{key}__probability"]
    pid_all = sus_arr[f"{key}__patient_id"].astype(str)
    for layer in range(emb_all.shape[1]):
        df_l = diag.partial_pc1_year_analysis(
            emb_all[:, layer, :], yr_all, prob_all, pid_all,
            spline_df=diag.LOGIT_SPLINE_DF, draws=BOOTSTRAP_DRAWS, seed=diag.BOOTSTRAP_SEED,
            run_refit_sensitivity=True, min_axis_cos=diag.AXIS_ALIGNMENT_MIN_COS,
        )
        df_l.insert(0, "dataset", ds_label)
        df_l.insert(1, "scaling", csv_scaling)
        df_l.insert(2, "stratum", csv_stratum)
        df_l.insert(3, "layer", layer)
        pc1_frames.append(df_l)

        ref_row = sus_corr_csv[
            (sus_corr_csv["build"] == csv_build) & (sus_corr_csv["scaling"] == csv_scaling) & (sus_corr_csv["layer"] == layer)
        ]
        if len(ref_row) == 1:
            rr = ref_row.iloc[0]
            sub_f = df_l[df_l["mode"] == "fixed_pc1"].set_index("metric")["estimate"]
            pc1_max_parity_diff = max(
                pc1_max_parity_diff,
                abs(float(sub_f.loc["pc1_var_share"]) - float(rr["pc1_var_share"])),
                abs(float(sub_f.loc["abs_pearson_r_year_linear"]) - float(rr["abs_pearson_r"])),
            )
            pc1_parity_count += 2

# FLChain future 1998-2003
fl_future_mask = fl_arr["eval_stratum"] == "1998-2003"
fl_emb_fut = fl_arr["target_embedding"][fl_future_mask]
fl_yr_fut = fl_arr_sample_yr[fl_future_mask]
fl_prob_fut = fl_arr["probability"][fl_future_mask]
fl_sid_fut = fl_arr["subject_id"][fl_future_mask].astype(str)
for layer in range(fl_emb_fut.shape[1]):
    df_l = diag.partial_pc1_year_analysis(
        fl_emb_fut[:, layer, :], fl_yr_fut, fl_prob_fut, fl_sid_fut,
        spline_df=diag.LOGIT_SPLINE_DF, draws=BOOTSTRAP_DRAWS, seed=diag.BOOTSTRAP_SEED,
        run_refit_sensitivity=True, min_axis_cos=diag.AXIS_ALIGNMENT_MIN_COS,
    )
    df_l.insert(0, "dataset", "flchain_h1095")
    df_l.insert(1, "scaling", "consistent")
    df_l.insert(2, "stratum", "1998-2003")
    df_l.insert(3, "layer", layer)
    pc1_frames.append(df_l)

pc1_year_df = pd.concat(pc1_frames, ignore_index=True)
pc1_year_df.to_csv(OUTPUT_DIR / "pc1_year_partial_logit.csv", index=False)
record_check("sus_pc1_year_parity_vs_work8", pc1_max_parity_diff, f"<= {diag.PARITY_ATOL} (n={pc1_parity_count})", pc1_max_parity_diff <= diag.PARITY_ATOL)
print(f"Part 2a partial PC1 x year: rows={len(pc1_year_df)} | parity_checks={pc1_parity_count} | max_diff={pc1_max_parity_diff:.2e}")

Part 2a partial PC1 x year: rows=1728 | parity_checks=72 | max_diff=2.00e-15


### FLChain `2002` vs. `2003` Spectrum Split (Full & `n = 46` Subsamples)

In [9]:
fl_by_year_csv = pd.read_csv(FLCHAIN_BY_YEAR_DIR / "layer_curves_by_year.csv")

idx_2002 = np.flatnonzero(fl_future_mask & (fl_arr_sample_yr == 2002))
idx_2003 = np.flatnonzero(fl_future_mask & (fl_arr_sample_yr == 2003))
idx_pool = np.flatnonzero(fl_future_mask & np.isin(fl_arr_sample_yr, [2002, 2003]))

split_rows = []
for layer in range(fl_arr["target_embedding"].shape[1]):
    scale_l = fl_arr["scale_target"][layer]
    for grp_name, grp_idx in [("2002_only", idx_2002), ("2003_only_full", idx_2003), ("2002_2003_pooled", idx_pool)]:
        emb_g = fl_arr["target_embedding"][grp_idx, layer, :]
        spec = lc.weighted_spectrum(emb_g, scale_l, np.ones(len(grp_idx), dtype=np.float64))
        for chart in ("raw", "standardized"):
            for metric in ("embedding_pr", "top1_var_share", "top5_var_share", "top10_var_share"):
                split_rows.append({
                    "layer": layer, "group": grp_name, "chart": chart, "metric": metric,
                    "estimate": float(spec[chart][metric]), "sd_across_subsamples": np.nan,
                    "rows": int(len(grp_idx)), "repeats": 1,
                })

    # n=46 subsamples of 2003 matched to 2002 sample size
    sub_vals = {("raw", m): [] for m in ("embedding_pr", "top1_var_share", "top5_var_share", "top10_var_share")}
    sub_vals.update({("standardized", m): [] for m in ("embedding_pr", "top1_var_share", "top5_var_share", "top10_var_share")})
    rng_layer = np.random.default_rng(diag.SUBSAMPLE_SEED + layer)
    for _ in range(int(SUBSAMPLE_REPEATS)):
        pick = rng_layer.choice(idx_2003, size=diag.SUBSAMPLE_N_SMALL, replace=False)
        spec_sub = lc.weighted_spectrum(
            fl_arr["target_embedding"][pick, layer, :], scale_l, np.ones(diag.SUBSAMPLE_N_SMALL, dtype=np.float64)
        )
        for chart in ("raw", "standardized"):
            for metric in ("embedding_pr", "top1_var_share", "top5_var_share", "top10_var_share"):
                sub_vals[(chart, metric)].append(float(spec_sub[chart][metric]))
    for (chart, metric), vals in sub_vals.items():
        split_rows.append({
            "layer": layer, "group": "2003_subsample_n46", "chart": chart, "metric": metric,
            "estimate": float(np.mean(vals)), "sd_across_subsamples": float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0,
            "rows": int(diag.SUBSAMPLE_N_SMALL), "repeats": int(SUBSAMPLE_REPEATS),
        })

fl_2002_2003_df = pd.DataFrame(split_rows)
fl_2002_2003_df.to_csv(OUTPUT_DIR / "flchain_2002_2003_split.csv", index=False)

l0_pool_pr = float(fl_2002_2003_df.query("layer == 0 and group == '2002_2003_pooled' and chart == 'raw' and metric == 'embedding_pr'")["estimate"].iloc[0])
csv_pool_pr = float(fl_by_year_csv.query("layer == 0 and sample_year_group == '2002-2003' and token == 'target' and chart == 'raw' and metric == 'embedding_pr'")["estimate"].iloc[0])
pool_diff = abs(l0_pool_pr - csv_pool_pr)
record_check("flchain_2002_2003_pooled_l0_pr_parity", pool_diff, f"<= {diag.PARITY_ATOL}", pool_diff <= diag.PARITY_ATOL)
l0_2002_pr = float(fl_2002_2003_df.query("layer == 0 and group == '2002_only' and chart == 'raw' and metric == 'embedding_pr'")["estimate"].iloc[0])
l0_2003_pr = float(fl_2002_2003_df.query("layer == 0 and group == '2003_only_full' and chart == 'raw' and metric == 'embedding_pr'")["estimate"].iloc[0])
print(f"FLChain 2002-2003 L0 raw PR: pooled={l0_pool_pr:.4f} (Work 8={csv_pool_pr:.4f}), 2002_only={l0_2002_pr:.4f}, 2003_full={l0_2003_pr:.4f}")

FLChain 2002-2003 L0 raw PR: pooled=1.0042 (Work 8=1.0042), 2002_only=3.3545, 2003_full=3.7616


## 6. Part 2b — Threshold-Free Continuous TCAV Saturation (`L0..L11`)

In [10]:
sus_geom_csv = pd.read_csv(SUS_LAYER_DIR / "layer_geometry.csv")
fl_geom_csv = pd.read_csv(FLCHAIN_LAYER_DIR / "layer_geometry.csv")

sat_frames = []
rng_share_max_diff = 0.0
rng_se_max_diff = 0.0
rng_parity_count = 0

# SUS builds: pooled + per single input year
for key, ds_label, csv_build, csv_scaling, pooled_stratum in sus_build_configs:
    emb_all = sus_arr[f"{key}__target_embedding"]
    grad_all = sus_arr[f"{key}__target_gradient"]
    scale_all = sus_arr[f"{key}__scale_target"]
    yr_all = sus_arr[f"{key}__year"].astype(np.int64)
    strata_list = [(pooled_stratum, np.ones(len(yr_all), dtype=bool))]
    for yr_val in sorted(np.unique(yr_all).tolist()):
        strata_list.append((str(yr_val), yr_all == yr_val))

    for layer in range(emb_all.shape[1]):
        for stratum_name, mask in strata_list:
            sat_df = diag.continuous_tcav_saturation_for_layer(
                emb_all[mask, layer, :], grad_all[mask, layer, :], scale_all[layer],
                direction_count=diag.DIRECTION_COUNT, direction_seed=diag.DIRECTION_SEED, top_axes=diag.TOP_K,
            )
            sat_df.insert(0, "dataset", ds_label)
            sat_df.insert(1, "scaling", csv_scaling)
            sat_df.insert(2, "stratum", stratum_name)
            sat_df.insert(3, "layer", layer)
            sat_frames.append(sat_df)

            if stratum_name == pooled_stratum:
                ref_sub = sus_geom_csv[
                    (sus_geom_csv["build"] == csv_build) & (sus_geom_csv["scaling"] == csv_scaling)
                    & (sus_geom_csv["stratum"] == stratum_name) & (sus_geom_csv["layer"] == layer)
                    & (sus_geom_csv["token"] == "target") & (sus_geom_csv["section"] == "random")
                ]
                for _, srow in sat_df.iterrows():
                    rk = f"{srow['chart']}|{srow['family']}"
                    hit_sh = ref_sub[(ref_sub["key"] == rk) & (ref_sub["metric"] == "interior_share")]
                    hit_se = ref_sub[(ref_sub["key"] == rk) & (ref_sub["metric"] == "interior_se")]
                    if len(hit_sh) == 1 and len(hit_se) == 1:
                        rng_share_max_diff = max(rng_share_max_diff, abs(float(srow["interior_share"]) - float(hit_sh["value"].iloc[0])))
                        rng_se_max_diff = max(rng_se_max_diff, abs(float(srow["interior_se"]) - float(hit_se["value"].iloc[0])))
                        rng_parity_count += 1

# FLChain strata
fl_tcav_strata = [
    ("1997_eval", fl_arr["eval_stratum"] == "1997_eval"),
    ("1998-2003", fl_arr["eval_stratum"] == "1998-2003"),
    ("2002-2003", (fl_arr["eval_stratum"] == "1998-2003") & np.isin(fl_arr_sample_yr, [2002, 2003])),
]
for yr_val in (1998, 1999, 2000, 2001, 2002, 2003):
    fl_tcav_strata.append((str(yr_val), (fl_arr["eval_stratum"] == "1998-2003") & (fl_arr_sample_yr == yr_val)))

for layer in range(fl_arr["target_embedding"].shape[1]):
    scale_l = fl_arr["scale_target"][layer]
    for stratum_name, mask in fl_tcav_strata:
        sat_df = diag.continuous_tcav_saturation_for_layer(
            fl_arr["target_embedding"][mask, layer, :], fl_arr["target_gradient"][mask, layer, :], scale_l,
            direction_count=diag.DIRECTION_COUNT, direction_seed=diag.DIRECTION_SEED, top_axes=diag.TOP_K,
        )
        sat_df.insert(0, "dataset", "flchain_h1095")
        sat_df.insert(1, "scaling", "consistent")
        sat_df.insert(2, "stratum", stratum_name)
        sat_df.insert(3, "layer", layer)
        sat_frames.append(sat_df)

        if stratum_name in ("1997_eval", "1998-2003"):
            ref_sub = fl_geom_csv[
                (fl_geom_csv["stratum"] == stratum_name) & (fl_geom_csv["layer"] == layer)
                & (fl_geom_csv["token"] == "target") & (fl_geom_csv["section"] == "random")
            ]
            for _, srow in sat_df.iterrows():
                rk = f"{srow['chart']}|{srow['family']}"
                hit_sh = ref_sub[(ref_sub["key"] == rk) & (ref_sub["metric"] == "interior_share")]
                hit_se = ref_sub[(ref_sub["key"] == rk) & (ref_sub["metric"] == "interior_se")]
                if len(hit_sh) == 1 and len(hit_se) == 1:
                    rng_share_max_diff = max(rng_share_max_diff, abs(float(srow["interior_share"]) - float(hit_sh["value"].iloc[0])))
                    rng_se_max_diff = max(rng_se_max_diff, abs(float(srow["interior_se"]) - float(hit_se["value"].iloc[0])))
                    rng_parity_count += 1

tcav_sat_df = pd.concat(sat_frames, ignore_index=True)
tcav_sat_df.to_csv(OUTPUT_DIR / "continuous_tcav_saturation.csv", index=False)
record_check("work8_exact_rng_interior_share_parity", rng_share_max_diff, f"<= {diag.RNG_PARITY_ATOL} (n={rng_parity_count})", rng_share_max_diff <= diag.RNG_PARITY_ATOL)
record_check("work8_exact_rng_interior_se_parity", rng_se_max_diff, f"<= {diag.R_IDENTITY_ATOL} (n={rng_parity_count})", rng_se_max_diff <= diag.R_IDENTITY_ATOL)
print(f"Part 2b continuous TCAV saturation: rows={len(tcav_sat_df)} | exact RNG parity pairs={rng_parity_count} | max_share_diff={rng_share_max_diff:.2e} | max_se_diff={rng_se_max_diff:.2e}")

Part 2b continuous TCAV saturation: rows=1512 | exact RNG parity pairs=360 | max_share_diff=0.00e+00 | max_se_diff=9.89e-17


### Equal-`n` Subsample Controls (`n = 168` & `n = 46`) at Layers `0`, `9`, and `11`

In [11]:
eq_frames = []
control_layers = (0, 9, 11)

# Build named candidate pools for equal-n comparison
candidate_pools = [
    ("sus_same_year", "consistent", "input_years_2011_2013", sus_arr["same_year_consistent__target_embedding"], sus_arr["same_year_consistent__target_gradient"], sus_arr["same_year_consistent__scale_target"], np.ones(len(sus_arr["same_year_consistent__year"]), dtype=bool)),
    ("sus_same_year", "consistent", "2011", sus_arr["same_year_consistent__target_embedding"], sus_arr["same_year_consistent__target_gradient"], sus_arr["same_year_consistent__scale_target"], sus_arr["same_year_consistent__year"] == 2011),
    ("sus_same_year", "consistent", "2013", sus_arr["same_year_consistent__target_embedding"], sus_arr["same_year_consistent__target_gradient"], sus_arr["same_year_consistent__scale_target"], sus_arr["same_year_consistent__year"] == 2013),
    ("sus_prospective", "consistent", "2011", sus_arr["prospective_consistent__target_embedding"], sus_arr["prospective_consistent__target_gradient"], sus_arr["prospective_consistent__scale_target"], sus_arr["prospective_consistent__year"] == 2011),
    ("sus_prospective", "consistent", "2013", sus_arr["prospective_consistent__target_embedding"], sus_arr["prospective_consistent__target_gradient"], sus_arr["prospective_consistent__scale_target"], sus_arr["prospective_consistent__year"] == 2013),
    ("flchain_h1095", "consistent", "1997_eval", fl_arr["target_embedding"], fl_arr["target_gradient"], fl_arr["scale_target"], fl_arr["eval_stratum"] == "1997_eval"),
    ("flchain_h1095", "consistent", "2001", fl_arr["target_embedding"], fl_arr["target_gradient"], fl_arr["scale_target"], (fl_arr["eval_stratum"] == "1998-2003") & (fl_arr_sample_yr == 2001)),
    ("flchain_h1095", "consistent", "2002", fl_arr["target_embedding"], fl_arr["target_gradient"], fl_arr["scale_target"], (fl_arr["eval_stratum"] == "1998-2003") & (fl_arr_sample_yr == 2002)),
    ("flchain_h1095", "consistent", "2003", fl_arr["target_embedding"], fl_arr["target_gradient"], fl_arr["scale_target"], (fl_arr["eval_stratum"] == "1998-2003") & (fl_arr_sample_yr == 2003)),
]

for ds_label, scaling, stratum_name, emb_src, grad_src, scale_src, mask in candidate_pools:
    n_avail = int(np.sum(mask))
    for n_sub in (diag.SUBSAMPLE_N_CONTRAST, diag.SUBSAMPLE_N_SMALL):
        if n_avail < n_sub:
            continue
        if n_sub == diag.SUBSAMPLE_N_SMALL and stratum_name not in ("1997_eval", "2001", "2002", "2003", "2011", "2013"):
            continue
        for layer in control_layers:
            if n_avail == n_sub:
                base = diag.continuous_tcav_saturation_for_layer(
                    emb_src[mask, layer, :], grad_src[mask, layer, :], scale_src[layer],
                    direction_count=diag.DIRECTION_COUNT, direction_seed=diag.DIRECTION_SEED, top_axes=diag.TOP_K,
                )
                sub_df = base[["chart", "family", "mean_tcav_binary_entropy_nats", "mean_abs_dev_from_half", "exact_endpoint_share", "interior_share"]].copy()
                for c in ("sd_tcav_binary_entropy_nats", "sd_abs_dev_from_half", "sd_exact_endpoint_share", "sd_interior_share"):
                    sub_df[c] = 0.0
                sub_df["subsample_n"] = n_sub
                sub_df["repeats"] = 1
                sub_df["source_rows"] = n_avail
            else:
                sub_df = diag.continuous_tcav_equal_n_subsamples(
                    emb_src[mask, layer, :], grad_src[mask, layer, :], scale_src[layer],
                    subsample_n=n_sub, repeats=SUBSAMPLE_REPEATS, subsample_seed=diag.SUBSAMPLE_SEED,
                    direction_count=diag.DIRECTION_COUNT, direction_seed=diag.DIRECTION_SEED, top_axes=diag.TOP_K,
                )
            sub_df.insert(0, "dataset", ds_label)
            sub_df.insert(1, "scaling", scaling)
            sub_df.insert(2, "stratum", stratum_name)
            sub_df.insert(3, "layer", layer)
            eq_frames.append(sub_df)

tcav_eq_df = pd.concat(eq_frames, ignore_index=True)
tcav_eq_df.to_csv(OUTPUT_DIR / "continuous_tcav_equal_n.csv", index=False)
print(f"Equal-n continuous TCAV rows={len(tcav_eq_df)}")

Equal-n continuous TCAV rows=288


## 7. Part 2c — Analytical Random-Gradient Null Floor ($c_{192}/\sqrt{n}$) & Non-Nested Dispersion Summary

In [12]:
# Check exact R_probability == R_log_odds in both layer_geometry.csv files
def max_geom_r_gap(geom_df: pd.DataFrame) -> float:
    sub = geom_df[(geom_df["section"] == "coherence") & (geom_df["metric"] == "resultant_length")].copy()
    parts = sub["key"].str.split("|", expand=True)
    sub["chart"] = parts[0]
    sub["gscale"] = parts[1]
    piv = sub.pivot_table(index=["build", "scaling", "stratum", "layer", "chart"], columns="gscale", values="value")
    return float(np.max(np.abs(piv["probability"] - piv["log_odds"])))

gap_sus = max_geom_r_gap(sus_geom_csv)
gap_fl = max_geom_r_gap(fl_geom_csv)
r_gap_max = max(gap_sus, gap_fl)
record_check("resultant_length_probability_equals_log_odds", r_gap_max, f"<= {diag.R_IDENTITY_ATOL}", r_gap_max <= diag.R_IDENTITY_ATOL)

disp_rows = []
fl_eval_y_arr = fl_y_all[fl_arr["eval_row_index"]].astype(np.float64)

def add_single_year_strata(ds_label: str, scaling_scheme: str, strata_masks: list[tuple[str, np.ndarray]], emb_grad_prob_y_scale):
    grad_all, prob_all, y_all, scale_all = emb_grad_prob_y_scale
    for stratum_name, mask in strata_masks:
        p = prob_all[mask].astype(np.float64)
        yv = y_all[mask].astype(np.float64)
        clipped = np.clip(p, diag.PROB_CLIP_EPS, 1.0 - diag.PROB_CLIP_EPS)
        lgt = np.log(clipped / (1.0 - clipped))
        prob_std = float(np.std(p, ddof=0))
        prob_iqr = float(np.percentile(p, 75) - np.percentile(p, 25))
        logit_std = float(np.std(lgt, ddof=0))
        logit_iqr = float(np.percentile(lgt, 75) - np.percentile(lgt, 25))
        prev = float(np.mean(yv))
        for layer in range(grad_all.shape[1]):
            scale_l = scale_all[layer]
            g_layer = grad_all[mask, layer, :]
            for chart, cscale in (("raw", np.ones_like(scale_l)), ("standardized", scale_l)):
                coh = geometry.coherence(g_layer * cscale)
                null_info = diag.analytical_null_resultant(coh["rows_used"], dimension=g_layer.shape[1])
                disp_rows.append({
                    "dataset": ds_label, "scaling_scheme": scaling_scheme, "layer": layer, "chart": chart,
                    "stratum": stratum_name, "rows_used": int(coh["rows_used"]), "rows_dropped": int(coh["rows_dropped"]),
                    "probability_resultant_length": float(coh["resultant_length"]),
                    "clt_expected_floor": float(null_info["clt_expected_floor"]),
                    "rms_floor": float(null_info["rms_floor"]),
                    "ratio_observed_to_clt_floor": float(coh["resultant_length"] / null_info["clt_expected_floor"]),
                    "prob_std": prob_std, "prob_iqr": prob_iqr, "logit_std": logit_std, "logit_iqr": logit_iqr,
                    "prevalence": prev, "status": "ok",
                })

for key, ds_label, _, scaling_scheme, _ in sus_build_configs:
    yr_v = sus_arr[f"{key}__year"].astype(np.int64)
    masks = [(str(u), yr_v == u) for u in sorted(np.unique(yr_v).tolist())]
    add_single_year_strata(
        ds_label, key, masks,
        (sus_arr[f"{key}__target_gradient"], sus_arr[f"{key}__probability"], sus_arr[f"{key}__label"], sus_arr[f"{key}__scale_target"]),
    )

fl_masks = [("1997_eval", fl_arr["eval_stratum"] == "1997_eval")] + [
    (str(u), (fl_arr["eval_stratum"] == "1998-2003") & (fl_arr_sample_yr == u)) for u in (1998, 1999, 2000, 2001, 2002, 2003)
]
add_single_year_strata(
    "flchain_h1095", "consistent", fl_masks,
    (fl_arr["target_gradient"], fl_arr["probability"], fl_eval_y_arr, fl_arr["scale_target"]),
)

# Explicit unavailable placeholder rows for Work 7 Scope A L11
for chart in ("raw", "standardized"):
    disp_rows.append({
        "dataset": "sus_work7_scope_a", "scaling_scheme": "consistent", "layer": 11, "chart": chart,
        "stratum": "scope_a_models_2007_2012", "rows_used": np.nan, "rows_dropped": np.nan,
        "probability_resultant_length": np.nan, "clt_expected_floor": np.nan, "rms_floor": np.nan,
        "ratio_observed_to_clt_floor": np.nan, "prob_std": np.nan, "prob_iqr": np.nan,
        "logit_std": np.nan, "logit_iqr": np.nan, "prevalence": np.nan,
        "status": "unavailable_no_per_row_probs",
    })

disp_df = pd.DataFrame(disp_rows)
disp_df.to_csv(OUTPUT_DIR / "gradient_null_and_dispersion.csv", index=False)

disp_corr_df = diag.non_nested_dispersion_summary(disp_df[disp_df["status"] == "ok"])
disp_corr_df.to_csv(OUTPUT_DIR / "gradient_dispersion_correlations.csv", index=False)
print(f"Part 2c gradient null & dispersion: rows={len(disp_df)} | correlation summary rows={len(disp_corr_df)} | R gap={r_gap_max:.2e}")

Part 2c gradient null & dispersion: rows=386 | correlation summary rows=480 | R gap=3.33e-16


## 8. Part 2d — Input-Feature Correlation-Matrix PR & Permutation Null vs. Same-Row Standardized Embedding PR

In [13]:
# 1) FLChain creatinine missingness table & 6-feature primary PR
df_fl_raw = probe.load_flchain_dataframe(FLCHAIN_CSV_PATH, diag.FLCHAIN_CSV_SHA256)
kept_fl, _ = probe.build_flchain_target(df_fl_raw, 1095)
record_check("flchain_subject_id_aligned_with_splits", bool(np.array_equal(kept_fl["subject_id"].to_numpy(), fl_sid_all)), "== True", bool(np.array_equal(kept_fl["subject_id"].to_numpy(), fl_sid_all)))

kept_eval_df = kept_fl.iloc[fl_arr["eval_row_index"]].copy()
kept_eval_df["eval_stratum"] = fl_arr["eval_stratum"]
kept_eval_df["stratum_label"] = np.where(kept_eval_df["eval_stratum"] == "1997_eval", "1997_eval", kept_eval_df["sample.yr"].astype(str))

miss_rows = []
for sname in ["1997_eval", "1998", "1999", "2000", "2001", "2002", "2003"]:
    sub = kept_eval_df[kept_eval_df["stratum_label"] == sname]
    n_tot = len(sub)
    n_non_null = int(sub["creatinine"].notna().sum())
    n_miss = int(n_tot - n_non_null)
    miss_rows.append({
        "stratum": sname, "rows": n_tot,
        "creatinine_non_null": n_non_null, "creatinine_missing": n_miss,
        "creatinine_missing_share": float(n_miss / n_tot) if n_tot else np.nan,
    })
miss_df = pd.DataFrame(miss_rows)
miss_df.to_csv(OUTPUT_DIR / "flchain_creatinine_missingness_by_year.csv", index=False)

cols_6 = ["age", "sex_male", "kappa", "lambda", "flc.grp", "mgus"]
cols_5 = ["age", "sex_male", "kappa", "lambda", "mgus"]
X_fl_6_ref = kept_fl.iloc[fl_ref_fit_idx][cols_6].to_numpy(dtype=np.float64)
X_fl_5_ref = kept_fl.iloc[fl_ref_fit_idx][cols_5].to_numpy(dtype=np.float64)

input_spec_rows = []

def _same_row_std_emb_prs(emb_sub: np.ndarray, scale_all: np.ndarray) -> dict[int, float]:
    out_pr = {}
    for layer in (0, 9, 11):
        spec = lc.weighted_spectrum(emb_sub[:, layer, :], scale_all[layer], np.ones(len(emb_sub), dtype=np.float64))
        out_pr[layer] = float(spec["standardized"]["embedding_pr"])
    return out_pr

for sname, mask in fl_tcav_strata:
    row_idx_sub = fl_arr["eval_row_index"][mask]
    X_ev_6 = kept_fl.iloc[row_idx_sub][cols_6].to_numpy(dtype=np.float64)
    emb_prs = _same_row_std_emb_prs(fl_arr["target_embedding"][mask], fl_arr["scale_target"])

    spec6 = diag.input_feature_spectrum(X_ev_6, X_ref_scale=X_fl_6_ref, log1p=False, perm_draws=PERM_NULL_DRAWS, perm_seed=diag.PERM_NULL_SEED)
    input_spec_rows.append({
        "dataset": "flchain_h1095", "scaling": "consistent", "stratum": sname,
        "feature_set": "6_features_excl_creatinine", "transform": "none",
        **spec6,
        "emb_std_pr_layer_0": emb_prs[0], "emb_std_pr_layer_9": emb_prs[9], "emb_std_pr_layer_11": emb_prs[11],
        "tabpfn_preprocessing_note": "raw feature correlation vs post-LayerNorm embedding after TabPFN power/quantiles/outlier clipping",
    })

    X_ev_5 = kept_fl.iloc[row_idx_sub][cols_5].to_numpy(dtype=np.float64)
    spec5 = diag.input_feature_spectrum(X_ev_5, X_ref_scale=X_fl_5_ref, log1p=False, perm_draws=PERM_NULL_DRAWS, perm_seed=diag.PERM_NULL_SEED)
    input_spec_rows.append({
        "dataset": "flchain_h1095", "scaling": "consistent", "stratum": sname,
        "feature_set": "5_features_excl_creatinine_and_flc_grp", "transform": "none",
        **spec5,
        "emb_std_pr_layer_0": emb_prs[0], "emb_std_pr_layer_9": emb_prs[9], "emb_std_pr_layer_11": emb_prs[11],
        "tabpfn_preprocessing_note": "raw feature correlation vs post-LayerNorm embedding after TabPFN power/quantiles/outlier clipping",
    })

# Sensitivity with creatinine + missingness indicator ONLY on 1997_eval, 1998..2001, pooled 1998-2001
ref_creat_med = float(kept_fl.iloc[fl_ref_fit_idx]["creatinine"].median())
fl_creat_7 = kept_fl[cols_6 + ["creatinine"]].copy()
fl_creat_7["creatinine_missing"] = fl_creat_7["creatinine"].isna().astype(np.float64)
fl_creat_7["creatinine_filled"] = fl_creat_7["creatinine"].fillna(ref_creat_med)
cols_8_ind = cols_6 + ["creatinine_filled", "creatinine_missing"]
X_fl_8_ref = fl_creat_7.iloc[fl_ref_fit_idx][cols_8_ind].to_numpy(dtype=np.float64)

fl_1998_2001_strata = [("1997_eval", fl_arr["eval_stratum"] == "1997_eval"),
                       ("1998-2001", (fl_arr["eval_stratum"] == "1998-2003") & np.isin(fl_arr_sample_yr, [1998, 1999, 2000, 2001]))] + [
    (str(u), (fl_arr["eval_stratum"] == "1998-2003") & (fl_arr_sample_yr == u)) for u in (1998, 1999, 2000, 2001)
]
for sname, mask in fl_1998_2001_strata:
    row_idx_sub = fl_arr["eval_row_index"][mask]
    emb_prs = _same_row_std_emb_prs(fl_arr["target_embedding"][mask], fl_arr["scale_target"])
    spec8 = diag.input_feature_spectrum(
        fl_creat_7.iloc[row_idx_sub][cols_8_ind].to_numpy(dtype=np.float64),
        X_ref_scale=X_fl_8_ref, log1p=False, perm_draws=PERM_NULL_DRAWS, perm_seed=diag.PERM_NULL_SEED,
    )
    input_spec_rows.append({
        "dataset": "flchain_h1095", "scaling": "consistent", "stratum": sname,
        "feature_set": "8_features_with_creatinine_ref_median_and_missing_ind", "transform": "none",
        **spec8,
        "emb_std_pr_layer_0": emb_prs[0], "emb_std_pr_layer_9": emb_prs[9], "emb_std_pr_layer_11": emb_prs[11],
        "tabpfn_preprocessing_note": "1997-2001 creatinine sensitivity only; 2002-2003 excluded due to 90-91% missingness",
    })

# 2) SUS exact Work 8 scale rows (316 same-year, 278 prospective) & eval rows (500 features)
w7_meta = np.load(WORK7_META_PATH, allow_pickle=False)
roles_2011 = pilot.reference_year_roles(population, 2011, 42)
same_year_scale_idx = np.flatnonzero((population["year"] == 2011) & (roles_2011["row_role"] == "sae_discovery"))
prospective_scale_idx = np.flatnonzero((population["year"] == 2011) & w7_meta["in_cohort"] & (w7_meta["role_2011"] == "sae_discovery"))
record_check("sus_same_year_scale_row_count", len(same_year_scale_idx), "== 316", len(same_year_scale_idx) == 316)
record_check("sus_prospective_scale_row_count", len(prospective_scale_idx), "== 278", len(prospective_scale_idx) == 278)

sus_X_all = population["X"].astype(np.float64)
for key, ds_label, _, _, pooled_stratum in [sus_build_configs[0], sus_build_configs[2]]:
    scale_idx = same_year_scale_idx if key == "same_year_consistent" else prospective_scale_idx
    X_ref_s = sus_X_all[scale_idx]
    ev_row_idx = sus_arr[f"{key}__eval_row_index"]
    yr_v = sus_arr[f"{key}__year"].astype(np.int64)
    emb_all = sus_arr[f"{key}__target_embedding"]
    scale_all = sus_arr[f"{key}__scale_target"]

    sus_strata = [(pooled_stratum, np.ones(len(yr_v), dtype=bool))] + [
        (str(u), yr_v == u) for u in sorted(np.unique(yr_v).tolist())
    ]
    for sname, mask in sus_strata:
        X_ev_s = sus_X_all[ev_row_idx[mask]]
        emb_prs = _same_row_std_emb_prs(emb_all[mask], scale_all)
        for do_log1p in (False, True):
            spec_s = diag.input_feature_spectrum(
                X_ev_s, X_ref_scale=X_ref_s, log1p=do_log1p, perm_draws=PERM_NULL_DRAWS, perm_seed=diag.PERM_NULL_SEED
            )
            input_spec_rows.append({
                "dataset": ds_label, "scaling": "consistent", "stratum": sname,
                "feature_set": "500_event_count_features", "transform": "log1p" if do_log1p else "none",
                **spec_s,
                "emb_std_pr_layer_0": emb_prs[0], "emb_std_pr_layer_9": emb_prs[9], "emb_std_pr_layer_11": emb_prs[11],
                "tabpfn_preprocessing_note": "raw event counts vs post-LayerNorm embedding after TabPFN internal transforms",
            })

raw_spec_df = pd.DataFrame(input_spec_rows)
raw_spec_df.to_csv(OUTPUT_DIR / "raw_input_feature_spectrum.csv", index=False)
print(f"Part 2d input feature spectrum: rows={len(raw_spec_df)}")

Part 2d input feature spectrum: rows=40


## 9. Aggregated Summary Figures

In [14]:
# Figure 1: Calibration actual vs held (Brier, log-loss / CITL, recall vs false-alarm rate)
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
w1_later = work1_cal_df[(work1_cal_df["stratum"] == "later_years") & (work1_cal_df["scaling"] == "consistent")]
for q, marker, lbl in [("actual", "o-", "actual"), ("held", "s--", "held")]:
    b = w1_later[(w1_later["quantity"] == q) & (w1_later["metric"] == "brier")].sort_values("reference_year")
    axes[0].plot(b["reference_year"], b["estimate"], marker, label=lbl)
    ll = w1_later[(w1_later["quantity"] == q) & (w1_later["metric"] == "log_loss")].sort_values("reference_year")
    axes[1].plot(ll["reference_year"], ll["estimate"], marker, label=lbl)
    rec = w1_later[(w1_later["quantity"] == q) & (w1_later["metric"] == "recall")].sort_values("reference_year")
    far = w1_later[(w1_later["quantity"] == q) & (w1_later["metric"] == "false_alarm_rate")].sort_values("reference_year")
    axes[2].plot(rec["reference_year"], rec["estimate"], marker, label=f"recall ({lbl})")
    axes[2].plot(far["reference_year"], far["estimate"], ":", marker="x", label=f"FAR ({lbl})")
axes[0].set_title("Work 1 Pooled Later Years: Brier"); axes[0].set_xlabel("Reference year"); axes[0].legend()
axes[1].set_title("Work 1 Pooled Later Years: Log-Loss"); axes[1].set_xlabel("Reference year"); axes[1].legend()
axes[2].set_title("Work 1 Pooled Later Years: Recall vs FAR"); axes[2].set_xlabel("Reference year"); axes[2].legend(fontsize=7)
fig.tight_layout()
fig.savefig(FIG_DIR / "calibration_actual_vs_held.png", dpi=150)
plt.close(fig)

# Figure 2: Partial PC1 x year (raw vs linear logit vs natural spline logit) across L0..L11
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
for ax, ds_name, title in [(axes[0], "sus_prospective", "SUS Prospective (L0..L11)"), (axes[1], "sus_same_year", "SUS Same-Year Consistent (L0..L11)")]:
    sub = pc1_year_df[(pc1_year_df["dataset"] == ds_name) & (pc1_year_df["scaling"] == "consistent") & (pc1_year_df["mode"] == "fixed_pc1")]
    for m, lbl in [
        ("eta_squared_year_cat", "Unadjusted eta^2(year)"),
        ("partial_eta2_year_cat_given_logit_linear", "Partial eta^2(year | linear logit)"),
        ("partial_eta2_year_cat_given_logit_spline", "Partial eta^2(year | spline logit)"),
        ("residual_var_after_logit_spline", "Residual 1 - R^2(spline logit)"),
    ]:
        s = sub[sub["metric"] == m].sort_values("layer")
        ax.plot(s["layer"], s["estimate"], marker="o", label=lbl)
    ax.set_title(title); ax.set_xlabel("Layer"); ax.set_ylim(-0.02, 1.05); ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(FIG_DIR / "pc1_year_partial_logit.png", dpi=150)
plt.close(fig)

# Figure 3: Continuous TCAV binary entropy across layers & input correlation-matrix PR vs embedding PR
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
for ds_name, st_name, lbl in [
    ("sus_same_year", "input_years_2011_2013", "SUS same-year (iso)"),
    ("sus_prospective", "input_years_2011_2013", "SUS prospective (iso)"),
    ("flchain_h1095", "1998-2003", "FLChain 1998-2003 (iso)"),
]:
    sub = tcav_sat_df[(tcav_sat_df["dataset"] == ds_name) & (tcav_sat_df["scaling"] == "consistent") & (tcav_sat_df["stratum"] == st_name) & (tcav_sat_df["chart"] == "raw") & (tcav_sat_df["family"] == "isotropic")].sort_values("layer")
    axes[0].plot(sub["layer"], sub["mean_tcav_binary_entropy_nats"], marker="o", label=lbl)
axes[0].set_title("Random-Direction TCAV Binary Entropy"); axes[0].set_xlabel("Layer"); axes[0].set_ylabel("Mean H(TCAV) [nats]"); axes[0].legend(fontsize=8)

bars = raw_spec_df[((raw_spec_df["dataset"] == "sus_same_year") & (raw_spec_df["stratum"] == "input_years_2011_2013") & (raw_spec_df["transform"] == "none")) | ((raw_spec_df["dataset"] == "flchain_h1095") & (raw_spec_df["stratum"] == "1998-2003") & (raw_spec_df["feature_set"] == "6_features_excl_creatinine"))]
x_pos = np.arange(len(bars))
axes[1].bar(x_pos - 0.2, bars["corr_matrix_pr"], width=0.2, label="Input corr-matrix PR")
axes[1].bar(x_pos, bars["perm_null_pr_mean"], width=0.2, label="Permutation null PR")
axes[1].bar(x_pos + 0.2, bars["emb_std_pr_layer_0"], width=0.2, label="Same-row L0 std PR")
axes[1].set_xticks(x_pos); axes[1].set_xticklabels(bars["dataset"], fontsize=8); axes[1].set_title("Input Feature PR vs Null & L0"); axes[1].legend(fontsize=7)
fig.tight_layout()
fig.savefig(FIG_DIR / "continuous_tcav_and_input_pr.png", dpi=150)
plt.close(fig)
print("Saved 3 aggregated figures.")

Saved 3 aggregated figures.


## 10. Checks & Run Record

In [15]:
# Verify minimum published SUS aggregate cell size
sus_w1_rows = work1_cal_df[(work1_cal_df["quantity"] == "actual") & (work1_cal_df["metric"] == "rows")]["estimate"]
min_sus_cell = int(min(
    float(sus_w1_rows.min()),
    float(tcav_sat_df[tcav_sat_df["dataset"].str.startswith("sus_")]["rows_used"].min()),
    float(raw_spec_df[raw_spec_df["dataset"].str.startswith("sus_")]["rows_used"].min()),
))
record_check("min_published_sus_cell_count", min_sus_cell, "> 0", min_sus_cell > 0)

checks_df = pd.DataFrame(CHECKS)
checks_df.to_csv(OUTPUT_DIR / "checks.csv", index=False)
if not checks_df["passed"].all():
    failed_names = checks_df.loc[~checks_df["passed"], "check"].tolist()
    raise AssertionError(f"Checks failed: {failed_names}")

run_end_utc = datetime.now(timezone.utc).isoformat()
wall_seconds = round(time.monotonic() - run_start_mono, 2)
peak_rss_mb = round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024.0, 2)

output_files = {}
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file() and p.name != "run_record.json":
        output_files[str(p.relative_to(OUTPUT_DIR))] = {
            "bytes": int(p.stat().st_size),
            "sha256": probe.file_sha256(p),
        }

run_record = {
    "run_start_utc": run_start_utc,
    "run_end_utc": run_end_utc,
    "wall_seconds": wall_seconds,
    "peak_rss_mb": peak_rss_mb,
    "git_head": git_head,
    "notebook_sha256": probe.file_sha256(Path("calibration_and_layer_diagnostics.ipynb")) if Path("calibration_and_layer_diagnostics.ipynb").exists() else None,
    "uncommitted_files": git_status_lines,
    "device": "cpu",
    "bootstrap_draws": int(BOOTSTRAP_DRAWS),
    "perm_null_draws": int(PERM_NULL_DRAWS),
    "subsample_repeats": int(SUBSAMPLE_REPEATS),
    "min_published_sus_cell_count": min_sus_cell,
    "checks_passed": int(checks_df["passed"].sum()),
    "checks_total": int(len(checks_df)),
    "input_sha256": INPUT_SHA256,
    "output_files": output_files,
}
(OUTPUT_DIR / "run_record.json").write_text(json.dumps(run_record, indent=2, sort_keys=True), encoding="utf-8")
print(f"Completed: wall_seconds={wall_seconds} | peak_rss_mb={peak_rss_mb} | checks={int(checks_df['passed'].sum())}/{len(checks_df)} | outputs={len(output_files)}")

Completed: wall_seconds=826.6 | peak_rss_mb=1279.0 | checks=19/19 | outputs=16


## Leitura descritiva dos resultados

Os testes de sensibilidade à entrada de ano nos mesmos pacientes comparam `actual` e `held` frente à mortalidade observada. Nas diferenças abaixo, Δ é `actual − held` e os colchetes mostram ICs pareados de 95%.

- **Calibração.** No Trabalho 1, entre os sete sistemas de janela móvel retrospectiva (`last_3`) de 2007 a 2013 avaliados até 2014, a referência 2007 em escala consistente teve Δ Brier +0,01000 [+0,00854 a +0,01160] e Δ log-loss +0,05098 [+0,04368 a +0,05890] nos anos posteriores. Sob o limiar compartilhado, os níveis `actual / held` foram 0,62857 / 0,41143 para recall e 0,29593 / 0,10129 para FAR. No Escopo A do Trabalho 7, para a referência 2007 na mesma escala, o Δ Brier foi +0,01000 [+0,00808 a +0,01204] e o Δ de calibração média foi +0,09243 [+0,09177 a +0,09317]. Log-loss e ICs de `held` estão indisponíveis nesse escopo. Em FLChain, no horizonte de 1.095 dias e no agregado 1998–2003 consistente, Δ Brier foi +0,00002 [-0,00020 a +0,00029] e Δ AUROC foi +0,00193 [-0,00116 a +0,00527]. `1997_eval` fica separado como nível descritivo. O inventário contou 7000 de 9964 ICs finitos que excluem zero, com referência aritmética de 498,20 a 5%.
- **PC1 e ano.** Na camada 11 do SUS prospectivo, a correlação absoluta com ano linear foi 0,8416. Para ano categórico, o η² parcial foi 0,8314 com ajuste linear pelo logit e 0,8468 com spline. Com a spline, a variância residual foi 0,7898, o acréscimo no R² total foi 0,6688 e a proporção de reajustes com deriva de eixo foi 6,0%. Em FLChain, o PR bruto da camada 0 foi 1,0042 para 2002–2003 combinados, 3,3545 em 2002, 3,7616 em 2003 e 3,6435 na média de subamostras de 2003 com 46 linhas.
- **Saturação do TCAV.** Em FLChain 1998–2003, para direções isotrópicas na representação bruta, a entropia média nas camadas 0 e 11 foi 0,4135 e 0,1202 nats, respectivamente. Na camada 11, o erro-padrão de Monte Carlo da entropia foi 0,0046, o desvio absoluto médio de 0,5 foi 0,4495 e 53,30% dos escores ficaram exatamente em 0 ou 1. Na mesma camada em 2003, a entropia média nos controles de 168 e 46 linhas foi 0,0676 e 0,0638 nats.
- **Resultante dos gradientes.** Na representação bruta de FLChain, na camada 11, $R$ variou de 0,9754 a 0,9932 entre anos individuais, com piso isotrópico aproximado de 0,0386 a 0,1473. A diferença máxima de $R$ entre probabilidade e log-odds foi `3.33067e-16`. A correlação de Spearman com o desvio-padrão da probabilidade foi -0,821 em 7 anos. As correlações usam anos individuais dentro de cada conjunto, escala e camada.
- **Atributos e embeddings nas mesmas linhas.** Em FLChain, faltava creatinina em 91,3% das linhas de 2002 e 90,2% de 2003. No agregado 1998–2003, os seis atributos da comparação principal tiveram PR 3,687 e nulo médio por permutação 5,981. Os PRs dos embeddings padronizados nas camadas 0, 9 e 11 foram 1,093 / 5,159 / 4,954. No SUS no mesmo ano, no agregado 2011–2013, o PR dos 500 atributos de contagem na origem foi 81,528 sem transformação e 79,690 com `log1p`, com nulos médios 99,679 e 99,682. Os PRs dos embeddings nas mesmas linhas foram 1,045 / 1,777 / 4,946. A comparação descreve estágios com pré-processamento diferente no TabPFN.